In [1]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import sqlite3

print("Libraries Loaded")

Libraries Loaded


In [2]:
# ============================================================
# LOAD DATA FROM SQLITE DATABASE
# ============================================================

db_path = r"C:\Users\acer\Downloads\Nifty 50 Stock Analysis\nifty50.db"

conn = sqlite3.connect(db_path)

df = pd.read_sql("SELECT * FROM stocks_data", conn)

conn.close()

# Convert date column
df["date"] = pd.to_datetime(df["date"])

# Remove invalid trading rows
df = df[df["volume"] > 0]

print("Data Loaded Successfully")
print("Dataset Shape:", df.shape)

df.head()

Data Loaded Successfully
Dataset Shape: (980, 7)


,date,stock,open,high,low,close,volume
0,2026-01-05,ADANIENT,2287.000000,2303.899902,2274.300049,2279.500000,696380
1,2026-01-06,ADANIENT,2288.899902,2290.399902,2248.100098,2259.100098,698294
2,2026-01-07,ADANIENT,2265.000000,2284.000000,2243.000000,2274.100098,529451
3,2026-01-08,ADANIENT,2275.000000,2275.899902,2205.000000,2214.000000,792800
4,2026-01-09,ADANIENT,2214.100098,2216.000000,2145.000000,2153.699951,1570376


In [3]:
# ============================================================
# DATA TRANSFORMATION
# ============================================================

pivot = df.pivot(
    index="date",
    columns="stock",
    values="close"
)

returns = pivot.pct_change().dropna()

print("Pivot Table Created")
print("Returns Calculated")

pivot.head()

Pivot Table Created
Returns Calculated


stock,ADANIENT,ADANIPORTS,APOLLOHOSP,ASIANPAINT,AXISBANK,BAJAJ-AUTO,BAJAJFINSV,BAJFINANCE,BHARTIARTL,BPCL,...,SBIN,SUNPHARMA,TATACONSUM,TATASTEEL,TCS,TECHM,TITAN,ULTRACEMCO,UPL,WIPRO
date,,,,,,,,,,,,,,,,,,,,,
2026-01-05,2279.500000,1493.000000,7083.0,2815.600098,1285.800049,9497.5,2039.199951,978.750000,2105.000000,367.532379,...,1005.549988,1728.900024,1182.099976,185.720001,3158.678711,1596.900024,4079.699951,12087.0,805.099976,256.673309
2026-01-06,2259.100098,1473.199951,7348.0,2845.800049,1293.800049,9661.0,2044.599976,977.349976,2105.300049,360.627167,...,1018.900024,1760.199951,1210.400024,186.199997,3197.669922,1601.800049,4111.799805,12204.0,799.299988,258.915436
2026-01-07,2274.100098,1465.300049,7447.5,2809.399902,1295.500000,9789.5,2031.900024,968.799988,2084.199951,358.098511,...,1007.150024,1782.599976,1212.599976,183.800003,3236.759277,1625.199951,4273.200195,12184.0,802.950012,263.984558
2026-01-08,2214.000000,1465.199951,7364.5,2786.500000,1286.800049,9760.5,2008.900024,971.950012,2066.300049,344.822968,...,998.000000,1760.699951,1197.400024,180.119995,3146.696289,1577.900024,4249.000000,12064.0,794.500000,255.601028
2026-01-09,2153.699951,1435.900024,7256.5,2825.500000,1272.000000,9562.5,1992.400024,959.599976,2027.099976,344.433960,...,1000.500000,1729.900024,1175.900024,178.399994,3150.526855,1582.199951,4201.799805,11937.0,771.700012,255.357315


In [4]:
# ============================================================
# FINANCIAL METRICS
# ============================================================

TRADING_DAYS = 252

avg_return = returns.mean() * TRADING_DAYS

risk = returns.std() * np.sqrt(TRADING_DAYS)

sharpe = avg_return / risk

metrics = pd.DataFrame({
    "stock": returns.columns,
    "avg_return": avg_return.values,
    "risk": risk.values,
    "sharpe": sharpe.values
})

print("Financial Metrics Calculated")

metrics.head()

Financial Metrics Calculated


,stock,avg_return,risk,sharpe
0,ADANIENT,-0.259383,0.641094,-0.404595
1,ADANIPORTS,0.456018,0.514716,0.885961
2,APOLLOHOSP,0.041220,0.256183,0.160899
3,ASIANPAINT,-1.923367,0.303927,-6.328375
4,AXISBANK,0.763436,0.342772,2.227242


In [5]:
# ============================================================
# MONTE CARLO PORTFOLIO SIMULATION
# ============================================================

mean_returns = returns.mean() * TRADING_DAYS
cov = returns.cov() * TRADING_DAYS

n_assets = len(mean_returns)

N = 5000

weights = np.random.random((N, n_assets))
weights = weights / weights.sum(axis=1)[:, None]

portfolio_returns = weights @ mean_returns.values

portfolio_vol = np.sqrt(
    np.einsum('ij,jk,ik->i', weights, cov.values, weights)
)

sharpe_ratio = portfolio_returns / portfolio_vol

ports = pd.DataFrame({
    "return": portfolio_returns,
    "volatility": portfolio_vol,
    "sharpe": sharpe_ratio
})

print("Monte Carlo Simulation Complete")

ports.head()

Monte Carlo Simulation Complete


,return,volatility,sharpe
0,-0.101385,0.155201,-0.653249
1,-0.092303,0.151264,-0.610209
2,-0.137221,0.150249,-0.913290
3,-0.141234,0.161562,-0.874180
4,-0.141958,0.154199,-0.920616


In [6]:
# ============================================================
# OPTIMAL PORTFOLIO
# ============================================================

max_idx = ports["sharpe"].idxmax()

optimal_weights = weights[max_idx]

optimal_portfolio = pd.DataFrame({
    "stock": returns.columns,
    "weight": optimal_weights
})

optimal_portfolio = optimal_portfolio.sort_values(
    "weight",
    ascending=False
)

print("Optimal Portfolio Identified")

optimal_portfolio.head(10)

Optimal Portfolio Identified


,stock,weight
18,HDFCBANK,0.048902
44,TECHM,0.048112
21,HINDALCO,0.046811
43,TCS,0.044677
40,SUNPHARMA,0.042427
30,LTIM,0.040762
14,DRREDDY,0.039932
12,COALINDIA,0.039840
25,INFY,0.037534
9,BPCL,0.036834


In [7]:
# ============================================================
# EXPORT DATA FOR POWER BI DASHBOARD
# ============================================================

metrics.to_csv("stock_metrics_summary.csv", index=False)

optimal_portfolio.to_csv("portfolio_allocation.csv", index=False)

print("Data Exported for Power BI Dashboard")

Data Exported for Power BI Dashboard
